In [1]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import paths, boto3
from sagemaker.core.helper.session_helper import get_execution_role, Session

region='us-east-1'
model_package_group_name='abalone'
model_version=1
data_bucket='omm-test-bucket'
project_path = 'models/abalone'
account='088461143167'
boto_session=boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker', region_name=region)
s3_resource = boto_session.resource('s3')
p=paths.Paths(bucket_name='omm-test-bucket', project_name='abalone', model_prefix='abalone')
sagemaker_session = Session(boto_session=boto_session)
role=get_execution_role(sagemaker_session)

[06/15/26 20:54:59] INFO     Found credentials from IAM Role: ec2-1                             ]8;id=5801665;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5801666;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [ ]:
# MonitoringScheduleConfig
{
'ScheduleConfig':{
    'ScheduleExpression':str, 
    'DataAnalysisStartTime':str, 
    'DataAnalysisEndTime':str
}, 
'MonitoringJobDefinition':{
    'BaselineConfig':{
        'BaseliningJobName':str, 
        'ConstraintsResource':{'S3Uri':str}, 
        'StatisticsResource':{'S3Uri':str}, 
    }, 
    'MonitoringInputs':[
        {
            'EndpointInput':{
                'EndpointName':str, 
                'LocalPath':str, 
                'S3InputMode':str, 
                'S3DataDistributionType':str, 
                'FeaturesAttribute':str, 
                'InferenceAttribute':str, 
                'ProbabilityAttribute':str, 
                'ProbabilityThresholdAttribute':float, 
                'StartTimeOffset':str, 
                'EndTimeOffset':str, 
                'ExcludeFeaturesAttribute':str
            }, 
            'BatchTransformInput':{
                'DataCapturedDestinationS3Uri':str, 
                'DatasetFormat':{
                    'Csv':{'Header':str}, 
                    'Json':{'Line':str}, 
                    'Parquet':{}
                }, 
                'LocalPath':str, 
                'S3InputMode':str, 
                'S3DataDistributionType':str, 
                'FeaturesAttribute':str, 
                'InferenceAttribute':str, 
                'ProbabilityAttribute':str, 
                'ProbabilityThresholdAttribute':float, 
                'StartTimeOffset':str, 
                'EndTimeOffset':str, 
                'ExcludeFeaturesAttribute':str

            }
        },
    ], 
    'MonitoringOutputConfig':{
        'MonitoringOutputs':[
            {
                'S3Output':{
                    'S3Uri':str, 
                    'LocalPath':str, 
                    'S3UploadMode':str
                    }
            },
        ], 
        'KmsKeyId':str
    }, 
    'MonitoringResources':{
        'ClusterConfig':{
            'InstanceCount':int, 
            'InstanceType':str, 
            'VolumeSizeInGB':int, 
            'VolumeKmsKeyId':str
        }
    }, 
    'MonitoringAppSpecification':{
        'ImageUri':str, 
        'ContainerEntrypoint':[str], 
        'ContainerArguments':list[str], 
        'RecordPreprocessorSourceUri':str, 
        'PostAnalyticsProcessorSourceUri':str
    }, 
    'StoppingCondition':{'MaxRuntimeInSeconds':int}, 
    'Environment':map, 
    'NetworkConfig':{
        'EnableInterContainerTrafficEncryption':bool, 
        'EnableNetworkIsolation':bool, 
        'VpcConfig':{'SecurityGroupIds':list[str], 'Subnets':list[str]}
    }, 
    'RoleArn':str
}, 
'MonitoringJobDefinitionName':str, 
'MonitoringType':str
}

In [2]:
schedules = sm_client.list_monitoring_schedules(EndpointName='abalone-endpoint')
for schedule in schedules['MonitoringScheduleSummaries']:
    print(f"schedule: {schedule['MonitoringScheduleName']}")

schedule: ab-dq


In [ ]:
def create_scheduled_data_quality_monitor(boto_session, role, name, deploy_type, schedule_expression, constraints_path, statistics_path, monitor_dir, endpoint_name=None, data_cature_dir=None, instance_type='ml.m5.large', volume_size_in_gb=20, max_runtime_in_seconds=1800):

    sm_client = boto_session.client('sagemaker', region_name='us-east-1')

    if deploy_type == 'realtime':
        monitoring_inputs=[{
            "EndpointInput": {
                "EndpointName": endpoint_name,
                "LocalPath": "/opt/ml/processing/input/endpoint"
            }
        }]
    else:
        monitoring_inputs=[{
            "BatchTransformInput": {
                "DataCapturedDestinationS3Uri": f"{data_cature_dir}/",
                "LocalPath": "/opt/ml/processing/input",
                "DatasetFormat": {
                    "Csv": {"Header": False}
                }
            }
        }]

    sm_client.create_monitoring_schedule(
        MonitoringScheduleName=name,
            MonitoringScheduleConfig={
            "ScheduleConfig": {
                "ScheduleExpression": schedule_expression# "cron(0 * ? * * *)"
            },
            "MonitoringJobDefinition": {
                "BaselineConfig": {
                    "ConstraintsResource": {"S3Uri": constraints_path},
                    "StatisticsResource": {"S3Uri": statistics_path}
                },
                "MonitoringInputs": monitoring_inputs,
                "MonitoringOutputConfig": {
                    "MonitoringOutputs": [{
                        "S3Output": {
                            "S3Uri": f'{monitor_dir}/reports',
                            "LocalPath": "/opt/ml/processing/output"
                        }
                    }]
                },
                "MonitoringResources": {
                    "ClusterConfig": {
                        "InstanceCount": 1,
                        "InstanceType": instance_type,
                        "VolumeSizeInGB": volume_size_in_gb
                    }
                },
                "MonitoringAppSpecification": {
                    "ImageUri": "156813124566.dkr.ecr.us-east-1.amazonaws.com/sagemaker-model-monitor-analyzer"
                },
                "RoleArn": role,
                "StoppingCondition": {"MaxRuntimeInSeconds": max_runtime_in_seconds}
            },
            "MonitoringType": "DataQuality"
        }
    )

In [ ]:
create_scheduled_data_quality_monitor(
    boto_session, 
    role, 
    'ab-dq', 
    'realtime', 
    "cron(0 * ? * * *)", 
    f'{p.dq_monitor_dir}/info/constraints.json', 
    f'{p.dq_monitor_dir}/info/statistics.json', 
    p.dq_monitor_dir, 
    endpoint_name='abalone-endpoint', 
    data_cature_dir=p.data_capture_dir, 
    instance_type='ml.m5.large', 
    volume_size_in_gb=20, 
    max_runtime_in_seconds=1800
)